### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

## Import necessary APIs
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [2]:
from langsmith import Client

client = Client()

## Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['512f0039-b5f7-4f79-96d4-e9f894c4bd60',
  '14bccfec-1d71-4228-b638-cf600fe2f652',
  '5ad40470-723e-4bca-880e-6015afaa48a6',
  '39bfcf91-2f72-415e-bde3-ab5c3faba926',
  '9325c751-eaa7-4621-8179-635086e65065'],
 'count': 5,
 'as_of': '2026-10-15T08:12:31.647046601Z'}

### Define Metrics (LLM As A Judge)


In [3]:
import openai
import groq
from langsmith import wrappers

## This is the default OpenAI wrapper
## openai_client = wrappers.wrap_openai(openai.OpenAI())

## The GROQ wrapper
openai_groq = wrappers.wrap_openai(openai.OpenAI(
      api_key = os.environ.get("GROQ_API_KEY"),
      base_url = "https://api.groq.com/openai/v1",
))

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict)->bool:
      
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      ## Import the groq client
      response = openai_groq.chat.completions.create(
            ## model="gpt-4o-mini",
            model = "openai/gpt-oss-120b", 
            temperature = 0,
            messages = [
                  {"role":"system","content": eval_instructions},
                  {"role":"user","content": user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"

## Concisions: checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [4]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "openai/gpt-oss-120b", instructions: str = default_instructions) -> str:
    """
    A Sample application to test the evaluations
    """
    return openai_groq.chat.completions.create(
        model = model,
        temperature = 0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

## Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [5]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target, ## Your AI system
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix = "gpt-oss-120b-chatbot"
)

View the evaluation results for experiment: 'gpt-oss-120b-chatbot-203a0e2a' at:
https://smith.langchain.com/o/f4d66530-564a-4074-a97b-bb3ab04bc2ec/datasets/6245b14a-a4a5-4cf7-829a-168651363b83/compare?selectedSessions=4ef970b8-170e-4392-97d7-d57318e60a0f




0it [00:00, ?it/s]

In [6]:
# ## Try to use different versions of LLMs and track them in Langsmith

# ## Call my_app for every datapoints
# def ls_target(inputs: str) -> dict:
#     return {"response": my_app(inputs["question"],model="gpt-4-turbo")}

# ## Run our evaluation
# experiment_results = client.evaluate(
#     ls_target, ## Your AI system
#     data = dataset_name,
#     evaluators = [correctness,concision],
#     experiment_prefix = "openai-4-turbo-chatbot"
# )

### Evaluation For RAG

In [7]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

## Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

## Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

## Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

## Set embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Add the document chunks to the "vector store" using HuggingFaceEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embedding = embeddings,
)

## With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k = 6)
retriever

C:\Users\dhruv\AppData\Local\Temp\ipykernel_20720\2888645753.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x000001F4D3FBA7B0>, search_kwargs={})

In [8]:
retriever.invoke("what are agents?")

[Document(id='fbc9ecd8-9128-43d5-88d2-01832cb09b9b', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [9]:
from langsmith import traceable
from langchain.chat_models import init_chat_model

## Initialise an LLM
llm = init_chat_model(model = "openai/gpt-oss-120b", model_provider = "groq")

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    
    ## Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""
    You are a helpful assistant who is good at analyzing source information and answering questions.       
    Use the following source documents to answer the user's questions.       
    If you don't know the answer, just say that you don't know.       
    Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke
    ai_msg = llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return {"answer":ai_msg.content,"documents":docs}

rag_bot("What is agents")

{'answer': 'Agents are autonomous software entities that perceive their environment, maintain internal memory, and make decisions to achieve goals. In LLM‑powered systems, the large language model serves as the agent’s brain, handling planning, subgoal decomposition, self‑reflection, and tool use. They interact with other agents and observations, using structured information (e.g., tree‑based environment data) to guide their actions.',
 'documents': [Document(id='fbc9ecd8-9128-43d5-88d2-01832cb09b9b', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver

### Dataset

In [10]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

## create the daatset and example in LAngsmith
dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name = dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

{'example_ids': ['2774af10-5c43-4114-a2c3-e864dcfddc7f',
  '965e9d61-bdf3-4c52-892f-73d9e81e8d38',
  '121117ac-52f0-4f11-ba11-544c32bc5adb'],
 'count': 3,
 'as_of': '2026-10-15T08:13:14.81257556Z'}

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [11]:
from typing_extensions import Annotated, TypedDict
## from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

## Correctness Output Schema

## Grade output schema
class CorrectnessGrade(TypedDict):
    ## Note that the order in the fields are defined is the order in which the model will generate them.
    ## It is useful to put explanations before responses because it forces the model to think through
    ## its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## Correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

## Import ChatGroq
grader_llm = ChatGroq(model = "openai/gpt-oss-120b", temperature = 0).with_structured_output(
    CorrectnessGrade, method="json_schema",strict=True
)

## Evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    ## Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])

    return grade["correct"]

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [12]:
## Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

## Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

## Relevance LLM
relevance_llm = ChatGroq(model = "openai/gpt-oss-120b", temperature = 0).with_structured_output(
    RelevanceGrade, method = "json_schema", strict = True
)

## Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [13]:
## Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

## Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

## Grounded LLM 
grounded_llm = ChatGroq(model = "openai/gpt-oss-120b", temperature = 0).with_structured_output(
    GroundedGrade, method = "json_schema", strict = True
)

## Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [14]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

## Retrieval LLM
retrieval_relevance_llm = ChatGroq(model = "openai/gpt-oss-120b", temperature = 0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])

    return grade["relevant"]

### Run the evaluation

In [15]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data = dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix = "rag-doc-relevance",
    metadata = {"version": "LCEL context, gpt-oss-120b-preview"},
)

## Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-a541cc86' at:
https://smith.langchain.com/o/f4d66530-564a-4074-a97b-bb3ab04bc2ec/datasets/680e7d94-0edc-460c-8a6a-7222b30dd99c/compare?selectedSessions=aead2f68-2e5a-46ad-a9bd-dcb99a31a00a




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator relevance> on run 01a13e9f-f7cc-7310-a7e4-ebcdab61ded8: RateLimitError("Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01knkfpsv7empswvk9a32ppfq3` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6786, Requested 1365. Please try again in 1.1325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}")
Traceback (most recent call last):
  File "d:\Udemy Material\4-Retrieval-Augmented-Generation\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1689, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "d:\Udemy Material\4-Retrieval-Augmented-Generation\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 370, i

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The ReAct framework itself focuses on interlea...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",False,True,True,True,0.921376,121117ac-52f0-4f11-ba11-544c32bc5adb,01a13e9f-6bda-71e1-afb7-f34ad4489191
1,What are the types of biases that can arise wi...,Few‑shot prompting can suffer from several sys...,[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,False,True,True,0.760023,2774af10-5c43-4114-a2c3-e864dcfddc7f,01a13e9f-8db0-7601-8f56-8e20258d6103
2,What are five types of adversarial attacks?,The five types of adversarial attacks are Toke...,[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,True,True,None,True,7.677498,965e9d61-bdf3-4c52-892f-73d9e81e8d38,01a13e9f-f7cc-7310-a7e4-ebcdab61ded8
